# PDF → CSV Conversion (Raw Data Source)

Extracts the homestay registry table from the government-issued PDF
(`updated_homestays_kalimpong.pdf`) into a clean, reproducible CSV.

Steps:
1. Extract table cells page-by-page using pdfplumber's structured table
   extraction (handles multi-line wrapped cells correctly, unlike raw
   text extraction).
2. Fix a font-encoding artifact present in this PDF.
3. Save to `../data/raw/kalimpong_homestays.csv`.


In [1]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import os
import re
import pandas as pd
import pdfplumber

In [2]:
# ==========================================
# CONFIGURATION
# ==========================================

RAW_PDF_FILE = "../data/dataset_reference/updated_homestays_kalimpong.pdf"
OUTPUT_FILE = "../data/raw/kalimpong_homestays.csv"

In [3]:
# ==========================================
# EXTRACT TABLE FROM PDF
# ==========================================

all_rows = []
with pdfplumber.open(RAW_PDF_FILE) as pdf:
    for page in pdf.pages:
        for table in page.extract_tables():
            for row in table:
                all_rows.append(row)

print(f"Raw rows extracted across all pages: {len(all_rows)}")

Raw rows extracted across all pages: 1973


In [4]:
# ==========================================
# CLEANING FUNCTIONS
# ==========================================

def first_cell_id(row):
    if not row or row[0] is None:
        return None
    last_line = str(row[0]).strip().split("\n")[-1].strip()
    return int(last_line) if last_line.isdigit() else None


def clean_cell(value):
    
    # Cleans a single extracted cell value.
    
    if value is None:
        return None
    text = str(value)
    text = text.replace("Yes", "y")
    text = " ".join(text.split("\n"))        # flatten wrapped multi-line cells
    text = text.replace("\u00bd", " 1/2")     # normalize the unicode half symbol
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else None

In [5]:
# ==========================================
# BUILD DATAFRAME
# ==========================================

records = []
for row in all_rows:
    sl_no = first_cell_id(row)
    if sl_no is None:
        continue
    cells_clean = [clean_cell(c) for c in row]
    records.append({
        "homestay_id": sl_no,
        "homestay_name": cells_clean[1],
        "owner_name": cells_clean[2],
        "category": cells_clean[3],
        "district": cells_clean[4],
        "block": cells_clean[5],
        "village": cells_clean[6],
        "owner_email": cells_clean[7],
        "owner_mobile": cells_clean[8],
    })

df = pd.DataFrame(records).sort_values("homestay_id").reset_index(drop=True)
print(f"Extracted {len(df)} records.")
df.head()

Extracted 1157 records.


,homestay_id,homestay_name,owner_name,category,district,block,village,owner_email,owner_mobile
0,1,Revere Homestay,Mr. Riwaj Pradhan,Silver,KALIMPONG,Municipality,"8 1/2 Mile, Kalimpong",riwajpradhan10@gmail.com,9800686780
1,2,Mansarover Homestay,Miss Tina Mani Gurung,Gold,KALIMPONG,Municipality,"Chandralok,",santabgurung53@gmail.com,9932234895
2,3,BETHANY HOMESTAY,ANUPAMA TAMANG,Silver,KALIMPONG,Kalimpong 1,.GRAHAMS HOME BLOCK B,wangchuck20199@gmial.com,8348993048
3,4,S3 HOMESTAY,SANGITA RAI,Silver,KALIMPONG,Kalimpong 1,UPPER ECHHEY DARA GAON KALIMPONG,sangitasankalp@gmail.com,9933410313
4,5,BAJARANGI HOMESTAY,KAMAL KUMAR SHARMA,Silver,KALIMPONG,Kalimpong 1,SINGI SAMALBONG KALIMPONG,bajrangihomestay@gmail.com,8670450557


In [6]:
# ==========================================
# SAVE CSV FILE
# ==========================================

df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(df)} records to {OUTPUT_FILE}")

Saved 1157 records to ../data/raw/kalimpong_homestays.csv
